
# **Output Parsers**

Output parsers are tools or methods used to extract structured data or information from unstructured or semi-structured textual outputs generated by language models or other natural language processing (NLP) systems. They are essential in converting raw text responses into structured formats that are easier to analyze, manipulate, and integrate into applications or workflows.


In [6]:
%%capture
# update or install the necessary libraries
!pip install --upgrade langchain langchain_community langchain_aws
!pip install --upgrade python-dotenv

In [7]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] =os.getenv('AWS_DEFAULT_REGION')

In [8]:
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="mistral.mistral-7b-instruct-v0:2",
    temperature=0.5
)

Sample customer review text with information to be extracted

Template string defining the prompt structure and variables to extract



In [9]:
customer_review = """\
While my experience at XYZ Corp has been largely positive, \
Our company offers a flexible work arrangement, including options for remote work. \
Employees are entitled to a minimum of three weeks of paid vacation per year.
Employee performance evaluations are conducted annually to assess job performance and set goals for improvement.\
"""

review_template = """\
For the following text, extract the following information:

Work_Life_Balance: What work arrangement options are available to employees? \
Answer True if yes, False if not or unknown.

Paid_Leaves(Weeks): How much paid vacation time are employees entitled to annually? \
If this information is not found, output -1.

performance_evaluation: How often are employee performance evaluations conducted?,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
Work_Life_Balance
Paid_Leaves(Weeks)
performance_evaluation

text: {text}
"""

In [11]:
# setting the review template for prompt
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)

In [12]:
# passing text as part of the prompt
messages = prompt_template.format_messages(text=customer_review)

In [33]:
messages

[HumanMessage(content='For the following text, extract the following information: and present the output in the format in json format\n\ntext: While my experience at XYZ Corp has been largely positive, Our company offers a flexible work arrangement, including options for remote work. Employees are entitled to a minimum of three weeks of paid vacation per year.\nEmployee performance evaluations are conducted annually to assess job performance and set goals for improvement.\n', additional_kwargs={}, response_metadata={})]

In [13]:
response = llm.invoke(messages)

In [14]:
response.content

' {\n"Work_Life_Balance": true,\n"Paid_Leaves(Weeks)": 3,\n"performance_evaluation": ["annually"]\n}'

In [ ]:
# You will get an error by running this line of code
# because'Paid_Leaves' is not a dictionary
# 'Paid_Leaves' is a string
response.content.get('Paid_Leaves(Weeks)')

AttributeError: 'str' object has no attribute 'get'

Define response schemas for structured output parsing


In [19]:
%%capture

%pip install langchain-classic
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser

In [20]:
#Define the ResponseSchema
Work_Life_Balance_schema = ResponseSchema(name="Work_Life_Balance",
                             description="What work arrangement options are available to employees? \
                             Answer True if yes, False if not or unknown.")
Paid_Leaves_schema = ResponseSchema(name="Paid_Leaves(Weeks)",
                                      description="How much paid vacation time are employees entitled to annually? \
                                      If this information is not found, output -1.")
performance_evaluation_schema = ResponseSchema(name="performance_evaluation",
                                    description="How often are employee performance evaluations conducted?,\
                                    and output them as a comma separated Python list.")

response_schemas = [Work_Life_Balance_schema, Paid_Leaves_schema,performance_evaluation_schema]

In [21]:
# configured the structured output parser
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [22]:
format_instructions = output_parser.get_format_instructions()

In [23]:
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"Work_Life_Balance": string  // What work arrangement options are available to employees?                              Answer True if yes, False if not or unknown.\n\t"Paid_Leaves(Weeks)": string  // How much paid vacation time are employees entitled to annually?                                       If this information is not found, output -1.\n\t"performance_evaluation": string  // How often are employee performance evaluations conducted?,                                    and output them as a comma separated Python list.\n}\n```'

In [28]:
review_template1 = """\
For the following text, extract the following information: and present the output in the format in json format

text: {text}
"""

In [29]:
# configuring output format
prompt = ChatPromptTemplate.from_template(template=review_template1)
messages = prompt.format_messages(text=customer_review, format_instructions=format_instructions)

In [32]:
messages

[HumanMessage(content='For the following text, extract the following information: and present the output in the format in json format\n\ntext: While my experience at XYZ Corp has been largely positive, Our company offers a flexible work arrangement, including options for remote work. Employees are entitled to a minimum of three weeks of paid vacation per year.\nEmployee performance evaluations are conducted annually to assess job performance and set goals for improvement.\n', additional_kwargs={}, response_metadata={})]

In [30]:
response = llm.invoke(messages)

In [31]:
print(response.content)

 {
"Company": "XYZ Corp",
"Work Arrangement": {
  "Flexible": true,
  "Remote Work": true
},
"Vacation": {
  "Minimum": 3,
  "Weeks": true
},
"Performance Evaluations": {
  "Frequency": "Annually"
}
}


In [ ]:
output_dict = output_parser.parse(response.content)

In [ ]:
type(output_dict)

In [ ]:
print(output_dict.get('Paid_Leaves(Weeks)'))

# **Let's Do an Activity**

## **Objective**

Practice using a structured output parser to extract specific information from text using a language model.

## **Scenario**

You are working on a project where you need to analyze customer feedback to extract key details such as sentiment, product mentions, and issues reported. You'll utilize a language model to process the feedback and a structured output parser to extract structured information.

## **Steps**

* Define a Prompt Template
* Prepare Sample Feedback
* Create Response Schema
* Format Instructions and Prompt Template
* Interact with the Model
* Parse and Display Results